# Scientific Programming with Python 
## (Winter 2025/26)

# Session 02: Arrays with Numpy

* Organization
* Python Recap & Update
* Numpy

# Organization


## Exercise groups

* tutor groups are full
* there are still people without group
  - if you are still interested in join the group "Still interested in a group"

## Tutor meetings

There will be initial tutor meetings this week. Meetings can be online or onsite - discuss with your tutor.  For in-person meetings, we have the following rooms:
* Wed., 12:00 - 14:00  Room: 93/E01
* Wed., 14:00 - 16:00  Room: 93/E12
* Thu., 12:00 - 14:00  Room: 35/E22

At the meeting:
* present your solutions to the exercises
* questions & answers
* feedback

# New exercise sheet

* covers `numpy`
* will be uploaded later today
* to be solved befor October 28th


# Numerical Computing with NumPy

## Introductory example

Python built-in collections like `list` offer a flexible way of storing and maniupulating data. As dicussed previously, collections usually just store references to objects. While this is every convenient when writing code, it comes with costs in performance in memory. 

Let's look at an example. Say we took one million measurements in an experiment:

In [ ]:
import random 
measurements = [random.randint(150, 200) for _ in range(1_000_000)]
print(measurements[:100])

Now want to compute the mean of it. We could do it in the following manner:

In [ ]:
def mean(values):
    accumulator = 0
    for value in values:
        accumulator += value
    mean_value = accumulator / len(values)
    return mean_value

print(mean(measurements))

This works quite smooth, but actually it is rather slow since Python has to rebind a new variable in every loop and then has to check whether the `+` operation is supported between the `accumulator` and the current `measurement`. This prevents it from trying to add together objects that can't be added, but in this case we are pretty sure that we are only dealing with integers.

In [ ]:
%timeit mean(measurements)

We can already achieve faster calculations when trying to use as many of Python's built in function, such as `sum`.

In [ ]:
%timeit sum(measurements) / len(measurements)

If we could tell the interpreter that we are only adding integers, we could skip all that typechecking and speed up the operation. For this purpose, `numpy` was invented.  

To use `numpy` we have to import it. The import is usually aliased as `np` so we have to type less later on. Aliasing things is only recommended if it is well established in the community of the respective package.

In [ ]:
import numpy as np

Numpy's standard datatype is the `ndarray` (which stands for n-dimensional array). In the simplest case, numpy array can be created from list.

In [ ]:
measurements_array = np.array(measurements)
measurements_array

In [ ]:
type(measurements_array)

Numpy arrays behave very similar to lists, but have a fixed datatype underneath. Numpy automatically notices that all our values are intergers and chooses the appropriate datatype. An integer that takes up 64 bits of memory. https://docs.scipy.org/doc/numpy-1.13.0/user/basics.types.html

In [ ]:
measurements_array.dtype

Indexing works just as with normal Python lists.

In [ ]:
print(measurements_array[0])
print(measurements_array[10:15])

Moreover, numpy offers a lot of routines for mathematical operations of arrays. Let's see if we acually gained something by using numpy.

In [ ]:
%timeit np.mean(measurements_array)

A significant speedup in comparision to the pure Python implementations! After convincing ourselfs that NumPy is useful, we have a more in depth look at the numpy array.

## Anatomy of arrays
Every array has a bunch of attributes that yield inforation about what it is.

### dtype

`.dtype` gives information about the data type. arrays can contain bools, ints, unsigned ints, floats or complex numbers of various byte sizes. They can also store strings or Python objects, but that has very few use cases.

In [ ]:
values = [0, 1, 2, 3, 4]
int_arr = np.array(values, dtype='int')
int_arr, int_arr.dtype

If the dtype does not match the given values, numpy will cast everything to that data type.

In [ ]:
bool_arr = np.array(values, dtype='bool')
bool_arr, bool_arr.dtype

If no explicit data type is given, numpy will choose the "smallest common denominator". In the following example, everything becomes a float, as ints can be represented as floats, but not vice versa.

In [ ]:
values = [0, 1, 2.5, 3, 4]
float_arr = np.array(values)
float_arr, float_arr.dtype

However, once the data type is set, everything will be coerced to that type.

In [ ]:
int_arr[1] = 2.5
int_arr, int_arr.dtype

These non-Python data types force us to again think about problems like overflow etc.

In [ ]:
values = [0, 1, 2, 3, 4]
uint_arr = np.array(values, dtype='uint8')
uint_arr, uint_arr.dtype

In [ ]:
uint_arr[1] += 255
uint_arr

...and can lead to some problems when comparing them to standard python types

In [ ]:
print(type(measurements_array[0]), type(183))

In [ ]:
val = 1.2 - 1.0
arr = np.array([val], dtype=np.float32)
print(f'{val} == {arr[0]} -> {val == arr[0]}')

For better comparisons, you can compare using an epsilon-value:

In [ ]:
epsilon = 1e-6  # 1*10^(-6); 0.000001
abs(arr[0] - val) < epsilon

Python documentation: [Floating-Point Arithmetic: Issues and Limitations](https://docs.python.org/3/tutorial/floatingpoint.html)

### shape and ndim
`.shape` is very important for keeping track of arrays with more than one dimension. It is a tuple with the number of elementns in each dimension. `.ndim` is just the number of dimensions in total. 

In [ ]:
values = [0, 1, 2, 3, 4]
one_dim_arr = np.array(values)
one_dim_arr

In [ ]:
one_dim_arr.shape

In [ ]:
one_dim_arr.ndim

In [ ]:
values = [[0, 1, 2, 3, 4]] * 3
two_dim_arr = np.array(values)
two_dim_arr

In [ ]:
[[0, 1, 2, 3, 4]] * 3

In [ ]:
two_dim_arr.shape

In [ ]:
two_dim_arr.ndim

In [ ]:
two_dim_arr[1,1]

In [ ]:
two_dim_arr[1,1] = 10
two_dim_arr

In [ ]:
values = [[[0, 1, 2, 3, 4]] * 3] * 6
three_dim_arr = np.array(values)
three_dim_arr

In [ ]:
three_dim_arr.shape

In [ ]:
three_dim_arr.ndim

### Other attributes

In [ ]:
two_dim_arr

In [ ]:
two_dim_arr.T

In [ ]:
print(dir(two_dim_arr))

## Creating arrays
We already saw how arrays can be created from Python lists (the same works with tuples). However, we often would like to create arrays directly, without creating Python objects. This can be accomplished by several utility functions.

The equivalent of `range`.

In [ ]:
np.arange(9)

In [ ]:
np.arange(start=2, stop=14, step=2)

Creating an array with a certain number of values in a certain interval.

In [ ]:
np.linspace(start=-5, stop=5, num=9)

An array containing zeros. The default `dtype` is `float`.

In [ ]:
np.zeros(5)

`np.zeros` takes a `shape` argument that lets us create multidimensional arrays.

In [ ]:
np.zeros((2, 3, 2))

The same goes for `ones`, `empty` and `full`.

In [ ]:
np.ones(shape=(2, 3, 2))

In [ ]:
# Corresponds to whatever was left in memory. Using zeros for initialising arrays is usually saver.
np.empty(shape=(2, 3, 2))

In [ ]:
# empty is faster than initialized arrays but usually doesn't make a difference
%timeit np.empty(shape=100_000)
%timeit np.ones(shape=100_000)

In [ ]:
np.full(shape=(2, 3, 2), fill_value=42)

### Exercise

Create a 3*3 array that solely consists of True

In [ ]:
np.ones((3, 3), dtype=bool)

### Random values
`np.random` contains a lot of functions to create arrays filled with random values of various probability distributions. It also can provide a random number generator (RNG):

In [ ]:
rng = np.random.default_rng()

rng.random((3, 3))

In [ ]:
rng.integers(0, 10, (5, 5))

Using ``np.integers`` and converting it to the boolean dtype, allows to create random boolean arrays!

### Exercise
 
Create a 5x5 array in which *statistically* 1/4 of items are False, all others being True.

You can use the method ``astype(bool)`` to convert an integer-array into a boolean-array

In [ ]:
rng.integers(0, 4, (5, 5)).astype(bool)

Note that in this array, there are only *on average* 1/4 elements of the elements True. To create an array where precisely 1/4 of elements are True, one would create an array with that many True's and ``shuffle`` it, as we'll see lateron.

### Repeating values

``np.repeat`` repeats elements of an array:

In [ ]:
np.repeat(3, 5)

In [ ]:
np.repeat([[1,2],[3,4]], 2, axis=1)

`np.tile` is another way of repeating values using NumPy.

In [ ]:
print('Repeat:', np.repeat([1, 2, 3], 3))
print('Tile:', np.tile([1, 2, 3], 3))

### Reshape

In [ ]:
a = np.arange(start=2, stop=14)
print(a.shape)
a

In [ ]:
b = a.reshape(3, 4)
b

-1 as axis automatically figures out the size of the respective dimension

In [ ]:
a.reshape(-1, 2)

### Example
We would like to create an 2D array where each row is [1, 2, 3] and it should have 10 rows.

In [ ]:
#np.repeat(np.arange(1, 4), 10).reshape(-1, 10).T
np.tile(np.arange(1, 4), 10).reshape(10, -1)

### Comparing Arrays

In [ ]:
epsilon = 0.000000000001
a = np.zeros((3,3))
a[0,0] += epsilon

b = np.zeros((3,3))
print(a)
print(b)

In [ ]:
a == b

In [ ]:
(a == b).all()

In [ ]:
c = np.array([])
d = np.array([1])
(c == d).all()

flaws with doing this: 
* if either a or b is empty and the other one contains a single element, this will return True. (the comparison a==b returns an empty array, for which the all-operator returns True)
* If a and b don't have the same shape and aren't broadcastable, then this approach will raise an error.

Instead, use numpys provided functions!

In [ ]:
np.array_equal(c, d)

In [ ]:
np.allclose(a, b)

In [ ]:
np.isclose(a, b)

## Masking
Logical arrays, i.e. arrays containing boolean values, can be used to index other arrays. These logical arrays are then called masks. This is especially useful to index based on logical conditions.

In [ ]:
# A simple integer array.
arr = np.arange(1, 6)
arr

In [ ]:
# A boolean array of the same shape as arr.
mask = np.array([True, False, True, False, True])
mask

Using the mask for indexing returns an array with only elements at positions where `mask` is `True`.  

In [ ]:
arr[mask]

Luckily for us, Operators in numpy work element-wise and return a boolean array:

In [ ]:
arr < 10

Because of this, we can use direct comparison as a mask:

In [ ]:
arr[arr < 3]

Maks can be used for assignment, which keeps the shape of the original array (thanks to *fancy indexing*, which will come up later)

### Exercise

Replace all odd numbers in the given array with -1

In [ ]:
arr = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
arr

In [ ]:
arr[arr%2==1] = -1
arr

## Mathematical operations
Numpy contains a lot of mathematical functions that operate on arrays in a vectorized manner. That means that they are applied to each element, without explicit for-loops. Vectorized functions are called `ufuncs` (universal functions) in Numpy.

### Standard arithmetic

In [ ]:
arr = np.arange(9)
arr

In [ ]:
arr * 3

In [ ]:
arr + (arr*2)

In [ ]:
arr - arr

In [ ]:
arr / arr

In [ ]:
arr * arr

In [ ]:
arr ** 2

Using `@` you can even do matrix multiplication. In the case of 1d arrays, this is the inner product between two vectors.

In [ ]:
arr @ arr

In [ ]:
# That's the same as
np.sum(arr * arr)

Using these operators/universal function is usually faster than writing the operations otherwise.

In [ ]:
var1 = lambda: np.repeat(np.arange(1, 4), 30).reshape(3, -1).T.flatten()
var2 = lambda: np.arange(3 * 30) % 3 + 1
var3 = lambda: np.array([[1, 2, 3] for _ in range(30)]).flatten()

%timeit var1()
%timeit var2()
%timeit var3()

### Some standard functions

In [ ]:
arr

In [ ]:
np.log(arr)

In [ ]:
np.exp(arr)

In [ ]:
np.sin(arr)

np.sign returns -1 for negative values, +1 for positive ones, and 0 for 0:

In [ ]:
np.sign(arr)

Always try to use vectorized ufuncs instead of explicit loops!

### Broadcasting
What happens if you try to add arrays of different shapes? Numpy will try to expand the arrays according to three rules and try to make their shapes match, so the operation can be applied elementwise. 

**1. Rule** If the arrays have different numbers of dimensions, the smaller shape is padded with ones on its left side.<br/>
            Example: (5 x 3) + (3) &rarr; (5 x 3) + (**1** x 3)<br/>
**2. Rule** If the number of the dimensions matches, but the size of a dimension does not, dimensions with the size of 1 are expanded.<br/>
            Example: (5 x 3) + (1 x 3) &rarr; (5 x 3) + (**5** x 3)<br/>
**3. Rule** If the shapes of the  arrays still defer after applying the Rule 1 and 2, a broadcasting error is raised.

The figure below gives an illustration (source https://jakevdp.github.io/PythonDataScienceHandbook/02.05-computation-on-arrays-broadcasting.html)

![](figures/broadcasting.png)

The Numpy documentation gives further insights https://docs.scipy.org/doc/numpy-1.14.0/user/basics.broadcasting.html.

In [ ]:
a = np.arange(15).reshape(5, 3)
a

In [ ]:
b = np.arange(3)
b

In [ ]:
a + b

Here is a case in which broadcasting fails.

In [ ]:
c = np.arange(4)
a + c

### Aggregation functions
Aggregation functions are functions that reduce the dimensionality of an array. They provide an `axis` argument, to specify which dimension to reduce.

In [ ]:
rng = np.random.default_rng()
two_dim_arr = rng.integers(0, high=20, size=(4, 4))
two_dim_arr

If just the array is passed, the aggregation operation is performed over the whole array.

In [ ]:
np.min(two_dim_arr)

The optional `axis` argument allows us to specify, which dimension should be aggregated. You can think of it as the operation being applied to all entries that are obtained by keeping the indices in all dimensions fixed except for the `axis` dimension.
Let's look at the result of the minimum operation with `axis=0`:

In [ ]:
np.min(two_dim_arr, axis=0)

The axis concept extends to more than one dimension

In [ ]:
three_dim_arr = rng.integers(0, high=20, size=(4, 4, 4))
three_dim_arr

In [ ]:
np.min(three_dim_arr, axis=0)

Here the entry at index `[0, 0]`, i.e. `5` is the minimum of the following values. 

In [ ]:
for i in range(4):
    print(three_dim_arr[i, 0, 0])

Let's demonstrate all axes again with another three-dimensional array:

In [ ]:
a = np.array([[[2,4],[6,9]],[[3,1],[7,8]],[[4,5],[9, 0]]])
a, a.shape

In [ ]:
np.min(a)

In [ ]:
np.min(a, axis=0)

setting the axis-argument is the same as going through all other axes of the respective array in turn, returning the respective aggregate for every combination of these.

In [ ]:
for i in range(a.shape[1]):
    for j in range(a.shape[2]):
        print(a[:, i, j])

For axis=1, we loop through axis 0 and axis 2:

In [ ]:
a

In [ ]:
np.min(a, axis=1)

In [ ]:
for i in range(a.shape[0]):
    for j in range(a.shape[2]):
        print(a[i, :, j])

...and finally, for axis 2 we loop through axis 0 and 1

In [ ]:
a

In [ ]:
np.min(a, axis=2)

In [ ]:
for i in range(a.shape[0]):
    for j in range(a.shape[1]):
        print(a[i, j, :])

The shape of the resulting array is simply the shape of the original array, leaving the specified axis out:

In [ ]:
mins = []
for i in range(a.shape[0]):
    for j in range(a.shape[1]):
        mins.append(min(a[i,j,:]))
np.array(mins).reshape([a.shape[0], a.shape[1]])

...however, of course, using numpy is much faster than looping over the array:

In [ ]:
def find_min_manual(arr):
    mins = []
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            mins.append(min(arr[i,j,:]))
    np.array(mins).reshape([arr.shape[0], arr.shape[1]])

%timeit find_min_manual(a)
%timeit np.min(a, axis=2)

#### np.nan

In [ ]:
np.nan == np.nan

In [ ]:
np.isnan(np.nan)

In [ ]:
a = np.r_[np.arange(5), np.repeat(0, 5)]
a

In [ ]:
b = a / a
b

In [ ]:
~np.isnan(b)

In [ ]:
b[~np.isnan(b)]

In [ ]:
np.divide(a, a, out=np.zeros(a.shape), where=(a!=0)) 
# at the positions where a!=0, make the division,
# at other indices use what's specified as "out"

### More than one Dimension

Aggregation functions can also aggregate more than one dimension at once.

In [ ]:
three_dim_arr = rng.integers(0, 10, (4, 2, 3))
three_dim_arr

In [ ]:
np.min(three_dim_arr, axis=(1, 2))

### Other aggregation functions.

In [ ]:
two_dim_arr

In [ ]:
np.max(two_dim_arr)

In [ ]:
np.max(two_dim_arr, axis=0)

In [ ]:
np.max(two_dim_arr, axis=1)

In [ ]:
np.sum(two_dim_arr)

In [ ]:
np.sum(two_dim_arr, axis=0)

In [ ]:
np.sum(two_dim_arr, axis=1)

Many of these function are also available as method on the array object.

In [ ]:
two_dim_arr.sum(axis=0)

### Flattening
We would like to convert any given array into a 1D array.

In [ ]:
a = np.arange(64).reshape((2,2,2,2,2,2))
a

In [ ]:
a.flatten()

Flatten vs. ravel:
* `flatten` always returns a copy
* `ravel` returns a contiguous view of the original array whenever possible.

## Advanced indexing
Numpy provides indexing methods that go beyond the indexing techniques known from standard Python sequences.


### Multidimensional indexing

You can use a colon to get all values from that dimensions.

In [ ]:
large_two_dim_arr = np.arange(81).reshape((9, 9))
large_two_dim_arr

In [ ]:
large_two_dim_arr[:, 1]

Standard slicing with `(start, stop, step)` works as expected.

In [ ]:
large_two_dim_arr[:, 1:3]

In [ ]:
large_two_dim_arr[:, 2:7:2]

Slices of an array are always `views`. That means, you just "view" the same chunk of meomory from a different perspective. This saves a lot of memory, but it means also that the original array will be changed, if you change the view.

In [ ]:
arr_slice = large_two_dim_arr[:, 1]
arr_slice[:] = 0
large_two_dim_arr

In [ ]:
large_two_dim_arr[:, 2] = 0
large_two_dim_arr

In [ ]:
l2 = np.copy(large_two_dim_arr)
l2[:, 6] = 0
large_two_dim_arr

In [ ]:
l2

If you need all values from several consecutive dimensions you can use ellipsis (`...`) as a shorthand.

In [ ]:
# Ellipsis is an actual Python object.
print(...)

In [ ]:
# np.stack joins arrays along a new axis.
four_dim_arr = np.stack((np.ones((3, 3, 3)), 
                         np.ones((3, 3, 3)) * 2, 
                         np.ones((3, 3, 3)) * 3, 
                         np.ones((3, 3, 3)) * 4))
four_dim_arr

In [ ]:
four_dim_arr.shape

In [ ]:
four_dim_arr[3, :, :, :]

In [ ]:
four_dim_arr[1,..., 1]

In [ ]:
four_dim_arr[..., 1]

### Fancy indexing
You can pass an array containing indices, this especially useful for drawing random items from an array.

In [ ]:
arr = np.arange(9) + 10
arr

In [ ]:
indices = np.array([1, 4, 5])
arr[indices]

The resulting array will reflect the shape of the index array.

In [ ]:
indices = np.array([[1, 4],
                    [5, 7]])
arr[indices]

You can index each dimension separately.

In [ ]:
two_dim_arr = np.arange(25).reshape(5, 5)
two_dim_arr

In [ ]:
x_indices = np.array([3, 4])
y_indices = np.array([1, 2])
two_dim_arr[x_indices, y_indices] # Corresponds to indexing at [3, 1] and [4, 2].

Using ``np.argsort``, you can sort the indices of an array, such that you can sort other arrays the same way (note that I'm using vstack only for demonstration purposes)

In [ ]:
a = rng.integers(0, 10, 10)
b = a**2
np.vstack((a, b))

In [ ]:
indices = a.argsort()
indices

In [ ]:
np.vstack((a[indices], b[indices]))

### Advanced Masking

In [ ]:
arr = np.arange(1, 7)
arr

Different masks can be combined using bitwise logical operators. These are the vectorized version of logical operators and should not be confused with `and`, `or` and `not` when evaluating the truth value of a whole object.

In [ ]:
smaller_or_equal_four = (arr <= 4)
smaller_or_equal_four   

In [ ]:
greater_two = (arr > 2)
greater_two

Bitwise and `&`.

In [ ]:
greater_two & smaller_or_equal_four

In [ ]:
# This does not work.
greater_two and smaller_or_equal_four

In [ ]:
arr

In [ ]:
arr[greater_two & smaller_or_equal_four]

Bitwise or using `|`.

In [ ]:
arr[greater_two | smaller_or_equal_four]

Bitwise xor using `^`.

In [ ]:
arr

In [ ]:
arr[greater_two ^ smaller_or_equal_four]

Bitwise negation using `~`.

In [ ]:
arr[~((arr < 2) ^ (arr > 2))]

In [ ]:
# Gives everything smaller or equal to 2.

arr[~greater_two]

In [ ]:
arr[~greater_two] = 2
arr

#### Using np.where

Using masking always changes the original array, whereas sometimes the original array should rather be unchanged. ``np.where`` figures out the indices of an array where the given condition is true.

In [ ]:
a = np.arange(9).reshape(3, 3)
a[a % 3 == 0] = 0
a

In [ ]:
a = np.arange(9).reshape(3, 3)
indices = np.where(a % 3 == 0)
indices

In [ ]:
b = np.ones((3, 3))
b[indices] = 0
b

``where`` can also be used to assign values to a new array:

In [ ]:
np.where(a % 3 == 0, 0, a)

In [ ]:
a

``np.argwhere`` returns the indices grouped by element:

In [ ]:
a = np.eye(4) * np.arange(16).reshape(4,4)
a

In [ ]:
np.argwhere(a)

In [ ]:
np.where(a)

## Extending arrays

### Adding new dimensions with `np.newaxis`

Instead of `np.newaxis`, `None` can be used.

In [ ]:
one_dim_arr = np.arange(5)
one_dim_arr, one_dim_arr.shape

In [ ]:
two_dim_arr = one_dim_arr[np.newaxis, :]
two_dim_arr, two_dim_arr.shape

In [ ]:
two_dim_arr = one_dim_arr[:, np.newaxis, None]
two_dim_arr, two_dim_arr.shape

Adding new dimensions is useful for example when Tensorflow is used to batch-inputs, but you want to provide a single datapoint for prediction:

In [ ]:
one_dim_arr[:, None]

### Removing dimensions

``arr.squeeze()`` removes dimensions of size 1:

In [ ]:
one_dim_arr = np.arange(5)
two_dim_arr = one_dim_arr[np.newaxis, :]
two_dim_arr, two_dim_arr.shape

In [ ]:
two_dim_arr.squeeze(), two_dim_arr.squeeze().shape

In [ ]:
a = np.arange(5).reshape(1, -1, 1, 1)
a

In [ ]:
a.squeeze()

### Combining arrays
There are many ways to combine existing arrays, like `np.append`, `np.concatenate` and `np.stack`. However, these operations always require the whole array to be copied. Therefore, it often makes more sense to allocate an array of the size you need later upfront and then just fill the respective parts.

In [ ]:
np.concatenate((np.arange(10), np.arange(10)[::-1]))

A quick and easy way to combine scalars and arrays is using ``np.r_``, with the desired arrays, lists, or numbers in square brackets:

In [ ]:
np.r_[2, 2, 2, np.arange(10), np.arange(10)[::-1], [0, 1, 2]]

``np.append`` uses concatenation internally:

In [ ]:
np.append(np.arange(10), np.arange(10))

For higher-dimensional arrays, other functions are useful:

In [ ]:
np.stack((np.arange(10), np.arange(10)))

There are also the functions ``np.vstack`` (row-wise-stacking) and ``np.hstack`` (column-wise-stacking):
* hstack is equivalent to concatenation along the second axis, except for 1-D arrays where it concatenates along the first axis
* vstack is equivalent to concatenation along the first axis after 1-D arrays of shape (N,) have been reshaped to (1,N).

In [ ]:
two_dim_arr = np.arange(16).reshape(4, -1)
two_dim_arr_2 = np.arange(16).reshape(4, -1) + 16
two_dim_arr, two_dim_arr_2

In [ ]:
np.hstack((two_dim_arr, two_dim_arr_2))

In [ ]:
np.vstack((two_dim_arr, two_dim_arr_2))

## Random values and random operations

### random.seed

If a random seed is set, the random-number-generator re-uses the same numbers over and over again. This is very useful for testing, but of course this takes any randomness out of anything, so it should not be used in final code:

In [ ]:
for _ in range(5):
    rng = np.random.default_rng(seed=0)
    print(rng.random(5))

To disable it, set a random seed of ``None``, in which case numpy generates random numbers using your system-randomness (or the system-time, ...)

In [ ]:
for _ in range(5):
    rng = np.random.default_rng(seed=None)
    print(rng.random(5))

### Shuffling arrays

``rng.shuffle`` shuffles an array among the first dimension. That means, a one-dimensional is completely shuffled, whereas for multidimensional arrays, the arrays from the second dimension on remain intact.

In [ ]:
a = np.arange(10)
rng.shuffle(a)
a

In [ ]:
Shuffling a multidimensional array just shuffles the first axis:

In [ ]:
a = np.arange(9).reshape(3, 3)
rng.shuffle(a)

To shuffle the array completely, you can flatten it and afterwards reshape it to its original shape:

In [ ]:
b = a.flatten()
rng.shuffle(b)
a = b.reshape(a.shape)
a

Note that ``np.shuffle`` shuffles the array in-place. To return a permutation, you'd use ``np.permutation``:

In [ ]:
a = np.arange(9).reshape(3, 3)
a

In [ ]:
rng.permutation(a)

In [ ]:
a

If you are dealing with different arrays you want to shuffle while keeping them matched to each other, it is more useful to shuffle the indices instead:

In [ ]:
a = np.arange(9)+1
b = a ** 2
np.vstack((a, b))

In [ ]:
order = rng.permutation(a.shape[0])
np.vstack((a[order], b[order]))

### random.choice

`rng.choice` generates a sub-array from a given 1D-array:

In [ ]:
a = np.arange(10)
rng.choice(a, size=5)

In [ ]:
rng.choice(a, size=5, replace=False)

In [ ]:
rng.choice(a, size=11, replace=False)  # this will fail

You can also specify probabilities which which to take certain elements. To generate another array where roughly a quarter of elements are True, you can use:

In [ ]:
rng.choice(np.array([True, False]), size=(5,5), p=[0.25, 0.75])

## The linear algebra module

Numpy provides a module for linear algebra: `numpy.linalg`
- Linear algebra routines for NumPy arrays (ndarray), backed by BLAS/LAPACK.
- Works with floats/complex numbers; integer inputs are cast to float.
- Operates on 2D arrays as matrices; vectors are 1D.
- Always check shapes and dtypes before calling linalg routines.

In [ ]:
A = np.array([[3.0, 2.0],
              [1.0, 2.0]])
b = np.array([2.0, 0.0])

print("A shape/dtype:", A.shape, A.dtype)
print("b shape/dtype:", b.shape, b.dtype)

Solving linear systems and least squares
- Solve $Ax = b$ with `np.linalg.solve` for square, nonsingular $A$.
- Over/underdetermined problems: `np.linalg.lstsq` minimizes $||Ax − b||^2$.
- Check residuals to assess fit quality.

In [ ]:
# Solve Ax = b
x = np.linalg.solve(A, b)
res = A @ x - b
print("x:", x)
print("residual norm:", np.linalg.norm(res))

In [ ]:
# Least squares fit y ≈ a + b*x
xdata = np.array([1.0, 2.0, 3.0])
ydata = np.array([1.0, 2.0, 2.7])
X = np.column_stack([np.ones_like(xdata), xdata])  # columns: [1, x]
coef, residuals, rank, svals = np.linalg.lstsq(X, ydata, rcond=None)
a_hat, b_hat = coef
print("least-squares parameters a, b:", a_hat, b_hat)
print("residual sum of squares:", residuals if residuals.size else 0.0)

Key decompositions (eig/eigh, SVD, QR)
- Eigen-decomposition: `np.linalg.eig` (general), `np.linalg.eigh` (symmetric/Hermitian).
- SVD: `np.linalg.svd` gives U, s, VT; useful for rank, pseudoinverse, PCA.
- QR: `np.linalg.qr` for orthogonal-triangular factorization (useful in solves/LS).

In [ ]:
# Symmetric eigen-decomposition
S = np.array([[2.0, 1.0],
              [1.0, 2.0]])
w, V = np.linalg.eigh(S)  # w sorted ascending
print("eigenvalues:", w)
print("check eigenpair 0:", np.allclose(S @ V[:, 0], w[0] * V[:, 0]))

In [ ]:
# SVD and reconstruction
M = np.array([[3.0, 1.0, 1.0],
              [-1.0, 3.0, 1.0]])
U, s, VT = np.linalg.svd(M, full_matrices=False)
M_rec = U @ np.diag(s) @ VT
print("SVD reconstruction close:", np.allclose(M, M_rec))

In [ ]:
# Reduced QR
Q, R = np.linalg.qr(M, mode="reduced")
print("QR check:", np.allclose(Q @ R, M), "Q orthogonal:", np.allclose(Q.T @ Q, np.eye(Q.shape[0])))

## Numpy print-options

In its standard options, numpy prints numbers only up to a certain accuracy, or consolidates the printing of arrays. ``np.set_printoptions`` can change that:

In [ ]:
rand_arr = rng.random((5,3))
rand_arr

In [ ]:
np.set_printoptions(precision=3)
rand_arr

In [ ]:
rand_arr = rand_arr/1e3
rand_arr

In [ ]:
np.set_printoptions(suppress=True, precision=6)
rand_arr

In [ ]:
arr = np.arange(90)
arr

In [ ]:
np.set_printoptions(threshold=10)
arr

## Further Readings

NumPy chapter from Jake VanderPlas's "Python Data Science Handbook" https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html

[Video tutorial from Scipy 2017](https://youtu.be/lKcwuPnSHIQ)


In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo('lKcwuPnSHIQ')